## Charger les données

In [1]:
import pandas as pd
from pathlib import Path

bronze_path = Path.cwd().parent / "data" / "bronze" / "parquet"
df_geolocation = pd.read_parquet (bronze_path/"olist_geolocation_dataset.parquet")

print("Dimensions :", df_geolocation.shape)

# Vérifier les types et valeurs non nulles
df_geolocation.info()

df_geolocation.isna().sum()

df_geolocation.head(5)

Dimensions : (1000163, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [2]:
df_geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


## Vérification les doublons

In [3]:

# Vérifier les lignes entièrement identiques
print(
    "Lignes dupliquées :",
    df_geolocation.duplicated().sum()
)

# Vérifier le nombre de codes postaux différents
print(
    "Codes postaux uniques :",
    df_geolocation["geolocation_zip_code_prefix"].nunique()
)

# Vérifier combien de codes postaux apparaissent plusieurs fois
print(
    "Codes postaux répétés :",
    (
        df_geolocation["geolocation_zip_code_prefix"]
        .value_counts() > 1
    ).sum()
)


Lignes dupliquées : 261831
Codes postaux uniques : 19015
Codes postaux répétés : 17972


## Vérification les coordonnées

In [4]:
# Vérifier les latitudes hors limites
print(
    "Latitudes invalides :",
    (~df_geolocation["geolocation_lat"].between(-90, 90)).sum()
)

# Vérifier les longitudes hors limites
print(
    "Longitudes invalides :",
    (~df_geolocation["geolocation_lng"].between(-180, 180)).sum()
)

# Afficher les statistiques
display(
    df_geolocation[
        ["geolocation_lat", "geolocation_lng"]
    ].describe()
)

Latitudes invalides : 0
Longitudes invalides : 0


,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


## Détecter les coordonnées hors du Brésil

In [5]:
# Vérifier les coordonnées situées hors de la zone approximative du Brésil
coordonnees_hors_bresil = (
    ~df_geolocation["geolocation_lat"].between(-34, 6)
    |
    ~df_geolocation["geolocation_lng"].between(-74, -34)
)

print(
    "Coordonnées hors du Brésil (zone approximative) :",
    coordonnees_hors_bresil.sum()
)

# Afficher quelques lignes suspectes
display(
    df_geolocation.loc[coordonnees_hors_bresil].head(10)
)

Coordonnées hors du Brésil (zone approximative) : 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


## Vérification les codes postaux

In [6]:
# Convertir les codes postaux en texte sur 5 caractères
zip_codes = (
    df_geolocation["geolocation_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

# Vérifier le format des codes postaux
zip_invalides = ~zip_codes.str.fullmatch(r"\d{5}")

print("Codes postaux invalides :", zip_invalides.sum())

Codes postaux invalides : 0
